In [0]:
from pyspark.sql.functions import col, trim, initcap, upper, to_date

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# CLIENTES
df_clientes_silver = (
    spark.table("bronze.clientes")
    .dropDuplicates(["id_cliente"])
    .withColumn("nombre", initcap(trim(col("nombre"))))
    .withColumn("pais", upper(trim(col("pais"))))
    .withColumn("email", trim(col("email")))
    .filter(col("id_cliente").isNotNull())
)

df_clientes_silver.write.mode("overwrite").saveAsTable("silver.clientes")

# PRODUCTOS
df_productos_silver = (
    spark.table("bronze.productos")
    .dropDuplicates(["id_producto"])
    .withColumn("nombre_producto", initcap(trim(col("nombre_producto"))))
    .withColumn("categoria", initcap(trim(col("categoria"))))
    .filter(col("precio_unitario") > 0)
)

df_productos_silver.write.mode("overwrite").saveAsTable("silver.productos")

# ORDENES
df_ordenes_silver = (
    spark.table("bronze.ordenes")
    .dropDuplicates(["id_orden"])
    .withColumn("fecha_orden", to_date(col("fecha_orden"), "yyyy-MM-dd"))
    .withColumn("estado_pago", upper(trim(col("estado_pago"))))
)

df_ordenes_silver.write.mode("overwrite").saveAsTable("silver.ordenes")

# DETALLE
df_detalle_silver = (
    spark.table("bronze.detalle_orden")
    .dropDuplicates(["id_detalle"])
    .filter(col("cantidad") > 0)
    .filter(col("precio_unitario") > 0)
    .withColumn("monto_linea", col("cantidad") * col("precio_unitario"))
)

df_detalle_silver.write.mode("overwrite").saveAsTable("silver.detalle_orden")

spark.sql("SHOW TABLES IN silver").show()